# Financial statement analysis

FCFF/FCFE construction for Organon and Sun Pharma, and a peer ratio benchmark.

In [1]:

import sys, warnings
sys.path.insert(0, r"/Users/shaan/Desktop/FAM/sunpharma-organon-merger-arbitrage")
warnings.filterwarnings("ignore")

from src import config as cfg, data, statements
import pandas as pd
pd.set_option("display.width", 160)

ogn_stmts = data.load_statements(cfg.TARGET)
sun_stmts = data.load_statements(cfg.ACQUIRER)


## FCFF, 5-year history

NOPAT uses yfinance's normalized marginal tax rate rather than the raw GAAP effective rate. Organon's raw effective rate swings from +56% to -52% year to year on valuation-allowance and one-off items - using it would make NOPAT non-comparable across years for reasons that have nothing to do with the operating business.

In [2]:

ogn_fcff = statements.build_fcff(ogn_stmts["income_stmt"], ogn_stmts["cashflow"])
ogn_fcff.round(1)


,2025-12-31,2024-12-31,2023-12-31,2022-12-31,2021-12-31
EBIT,929000000.0,1.327000e+09,1.200000e+09,1.544000e+09,NaN
normalized_tax_rate,0.2,2.000000e-01,2.000000e-01,2.000000e-01,NaN
NOPAT,733910000.0,1.048330e+09,9.480000e+08,1.261448e+09,NaN
D&A,361000000.0,2.770000e+08,2.360000e+08,2.120000e+08,NaN
CapEx,-316000000.0,-3.510000e+08,-2.610000e+08,-4.270000e+08,NaN
Delta_NWC,-247000000.0,-2.780000e+08,-1.550000e+08,-4.520000e+08,NaN
FCFF,531910000.0,6.963300e+08,7.680000e+08,5.944480e+08,NaN


In [3]:

sun_fcff = statements.build_fcff(sun_stmts["income_stmt"], sun_stmts["cashflow"])
sun_fcff.round(1)


,2026-03-31,2025-03-31,2024-03-31,2023-03-31
EBIT,1.545779e+11,1.398349e+11,1.132636e+11,9.580430e+10
normalized_tax_rate,2.000000e-01,2.000000e-01,1.000000e-01,1.000000e-01
NOPAT,1.182521e+11,1.116482e+11,9.853933e+10,8.717348e+10
D&A,2.937850e+10,2.575390e+10,2.556790e+10,2.530430e+10
CapEx,-3.609370e+10,-2.128580e+10,-2.201810e+10,-2.085580e+10
Delta_NWC,-1.267790e+10,-3.235600e+09,1.062130e+10,-5.661820e+10
FCFF,9.885899e+10,1.128807e+11,1.127104e+11,3.500378e+10


## Calendarisation

Organon's fiscal year ends in December; Sun Pharma's ends in March. Directly comparing "latest annual" figures would compare different economic periods. Organon's quarterly data is complete for the trailing four quarters, so a TTM figure is built there. Sun Pharma's quarterly series has a reporting gap (Sep-2025 quarter absent from the feed), so its FY2026 (Apr-2025 to Mar-2026) annual figure is used instead - the two reference periods overlap for three quarters and are offset by one, which is disclosed here rather than forced into an artificial alignment.

In [4]:

ogn_q = ogn_stmts["quarterly_income_stmt"]
ttm_cols = [c for c in ogn_q.columns if c >= pd.Timestamp("2025-07-01")][:4]
ogn_ttm_revenue = ogn_q.loc["Total Revenue", ttm_cols].sum()
ogn_ttm_ebit = ogn_q.loc["EBIT", ttm_cols].sum()

print("OGN TTM window:", [c.date() for c in ttm_cols])
print(f"OGN TTM revenue : ${ogn_ttm_revenue/1e9:.3f}B")
print(f"OGN TTM EBIT    : ${ogn_ttm_ebit/1e9:.3f}B")

sun_fy2026 = [c for c in sun_stmts["income_stmt"].columns if c.year == 2026][0]
sun_fy_revenue = sun_stmts["income_stmt"].loc["Total Revenue", sun_fy2026]
sun_fy_ebit = sun_stmts["income_stmt"].loc["EBIT", sun_fy2026]

print(f"\nSun Pharma FY2026 (period ended {sun_fy2026.date()})")
print(f"Sun FY revenue  : Rs {sun_fy_revenue/1e7:,.0f} Cr")
print(f"Sun FY EBIT     : Rs {sun_fy_ebit/1e7:,.0f} Cr")


OGN TTM window: [datetime.date(2026, 6, 30), datetime.date(2026, 3, 31), datetime.date(2025, 12, 31), datetime.date(2025, 9, 30)]
OGN TTM revenue : $6.127B
OGN TTM EBIT    : $0.959B

Sun Pharma FY2026 (period ended 2026-03-31)
Sun FY revenue  : Rs 58,220 Cr
Sun FY EBIT     : Rs 15,458 Cr


## Ratio benchmark vs peers

US peers (Merck, Pfizer, Viatris, Teva) benchmark Organon; Indian peers (Cipla, Dr Reddy's, Lupin) benchmark Sun Pharma. Each firm's own most recent annual figures - no cross-currency conversion is applied since every ratio here is a same-currency, same-firm margin or coverage figure.

In [5]:

def latest_ratios(ticker):
    stmts = data.load_statements(ticker)
    r = statements.ratio_snapshot(stmts["income_stmt"], stmts["balance_sheet"], stmts["cashflow"])
    return r.iloc[:, 0]

us_set = [cfg.TARGET] + cfg.US_PEERS
in_set = [cfg.ACQUIRER] + cfg.IN_PEERS

us_table = pd.DataFrame({t: latest_ratios(t) for t in us_set}).T
in_table = pd.DataFrame({t: latest_ratios(t) for t in in_set}).T

print("US pharma peer set (USD, most recent annual filing)")
us_table[["revenue","ebit_margin","ebitda_margin","net_margin","interest_coverage","debt_to_equity","roic"]].round(3)


US pharma peer set (USD, most recent annual filing)


,revenue,ebit_margin,ebitda_margin,net_margin,interest_coverage,debt_to_equity,roic
OGN,6.216000e+09,0.149,0.208,0.030,1.843,11.495,0.078
MRK,6.501100e+10,0.345,0.435,0.281,16.525,0.938,0.174
PFE,6.257900e+10,0.163,0.268,0.124,3.815,0.740,0.054
VTRS,1.429990e+10,-0.223,-0.028,-0.246,-6.776,0.999,-0.087
TEVA,1.725700e+10,0.124,0.182,0.082,2.335,2.161,0.068


In [6]:

print("Indian pharma peer set (INR, most recent annual filing)")
in_table[["revenue","ebit_margin","ebitda_margin","net_margin","interest_coverage","debt_to_equity","roic"]].round(3)


Indian pharma peer set (INR, most recent annual filing)


,revenue,ebit_margin,ebitda_margin,net_margin,interest_coverage,debt_to_equity,roic
SUNPHARMA.NS,5.822011e+11,0.266,0.316,0.197,45.610,0.055,0.139
CIPLA.NS,2.771169e+11,0.190,0.229,0.140,109.307,0.018,0.120
DRREDDY.NS,3.359330e+11,0.174,0.236,0.128,15.665,0.205,0.105
LUPIN.NS,2.748754e+11,0.265,0.315,0.194,17.661,0.295,0.203


## Leverage

Organon carries the debt load from its 2021 Merck spinoff. Net debt / EBITDA is a more stable read than debt-to-equity here, since Organon's equity base has been small and volatile (briefly negative in 2022), which makes D/E swing wildly for reasons unrelated to the actual debt burden.

In [7]:

ogn_ebitda_hist = ogn_stmts["income_stmt"].loc["EBITDA"]
ogn_debt_hist = ogn_stmts["balance_sheet"].loc["Total Debt"]
ogn_cash_hist = ogn_stmts["balance_sheet"].loc["Cash And Cash Equivalents"]
ogn_net_debt_hist = ogn_debt_hist - ogn_cash_hist

leverage = pd.DataFrame({
    "EBITDA": ogn_ebitda_hist,
    "net_debt": ogn_net_debt_hist,
    "net_debt_to_ebitda": ogn_net_debt_hist / ogn_ebitda_hist,
}).T
leverage.round(2)


,2025-12-31,2024-12-31,2023-12-31,2022-12-31,2021-12-31
EBITDA,1.290000e+09,1.604000e+09,1.436000e+09,1.756000e+09,NaN
net_debt,8.070000e+09,8.205000e+09,8.067000e+09,8.207000e+09,NaN
net_debt_to_ebitda,6.260000e+00,5.120000e+00,5.620000e+00,4.670000e+00,NaN


In [8]:

peer_multiples = pd.DataFrame({
    "ticker": us_set,
    "net_debt_usd_bn": [(us_table.loc[t, "net_debt"]) / 1e9 for t in us_set],
    "debt_to_equity": us_table["debt_to_equity"].values,
})
peer_multiples


,ticker,net_debt_usd_bn,debt_to_equity
0,OGN,8.0700,11.494681
1,MRK,34.7740,0.937897
2,PFE,62.8190,0.739639
3,VTRS,13.3756,0.999096
4,TEVA,13.5380,2.161062


In [9]:

summary = pd.DataFrame({
    "OGN_FCFF_TTM_proxy_usd_mn": [ogn_fcff.loc["FCFF"].iloc[0] / 1e6],
    "OGN_net_debt_to_ebitda_latest": [leverage.loc["net_debt_to_ebitda"].iloc[0]],
    "SUN_FCFF_latest_fy_inr_cr": [sun_fcff.loc["FCFF"].iloc[0] / 1e7],
}).T
summary.columns = ["value"]
summary.to_csv(cfg.DATA_FINAL / "statement_summary.csv")
summary


,value
OGN_FCFF_TTM_proxy_usd_mn,531.910000
OGN_net_debt_to_ebitda_latest,6.255814
SUN_FCFF_latest_fy_inr_cr,9885.899350
